# Module B Implementation and Optimization Report

> CS 432 Databases - Assignment 2 (Module B)

This report documents implementation evidence for SubTask 1 to SubTask 5:
- SubTask 1: Local environment setup and data integrity
- SubTask 2: Session-validated API and UI
- SubTask 3: RBAC and security logging
- SubTask 4: SQL indexing and query optimization
- SubTask 5: Before/after benchmarking and EXPLAIN analysis

## 1) Schema Design and Data Integrity (SubTask 1)

Core/project separation used in this module:
- Core identity/auth tables: `Member`, `AuthCredential`
- Core relation table: `Follow`
- Project content tables: `Post`, `Comment`

Integrity design choices:
- Credentials are not duplicated in project tables (stored only in `AuthCredential`).
- Foreign keys with cascade policies maintain consistency during create/delete operations.
- Constraint and trigger rules enforce data validity and business rules.

Member lifecycle requirement coverage:
- Admin member creation writes to both `Member` and `AuthCredential`.
- Member deletion cascades through related mappings/content as defined by schema constraints.

## 2) Security, Session Validation, and RBAC (SubTask 2 and 3)

Session validation:
- All protected endpoints validate a local JWT session token (`session-token` header).
- Invalid, missing, or expired sessions return `401`.

RBAC model:
- Admin users: member management and log inspection.
- Regular users: can modify only their own portfolio/posts/comments.
- Unauthorized cross-user modification attempts return `403`.

Security logging:
- API audit file: `Module_B/logs/audit.log`.
- DB-side write tracking: `ApiWriteLog` + triggers classify writes as `API` or `DIRECT_DB`.
- This makes direct SQL updates outside API/session validation easily identifiable as unauthorized.

## 3) SQL Indexing and Query Optimization (SubTask 4)

Frequently accessed / benchmarked endpoints:
- `GET /posts` (feed listing with filtering and ordering)
- `GET /posts/{post_id}/comments` (hotspot query under high-comment post)

Indexing strategy applied to API query clauses:
1. `idx_post_active_postdate_postid ON Post(IsActive, PostDate DESC, PostID DESC)`
   - Targets: `WHERE p.IsActive = TRUE` + `ORDER BY p.PostDate DESC, p.PostID DESC`
2. `idx_comment_post_active_date ON Comment(PostID, IsActive, CommentDate ASC)`
   - Targets: `WHERE c.PostID = ? AND c.IsActive = TRUE` + `ORDER BY c.CommentDate ASC`

Both are project-table indexes aligned to actual WHERE/JOIN/ORDER BY patterns.

## 4) Interactive Benchmarking and EXPLAIN Evidence (SubTask 5)

This section is now fully interactive inside this notebook.

Workflow executed in cells below:
1. Run tests without indexing (baseline stage).
2. Identify slow endpoints automatically from baseline SQL timing.
3. Create indexes only for the identified slow endpoints.
4. Rerun tests and compare before vs after (SQL, API, EXPLAIN).

Run these cells in order.

Prerequisites:
- Start API server from Module_B/app:
  - `uvicorn main:app --reload --port 8001`
- Ensure database is initialized from `sql/schema.sql` and `sql/sample_data.sql`.

Interpretation note:
- API-level speedups can be smaller than SQL-level speedups because API timing includes auth, serialization, and network overhead.
- EXPLAIN rows (`type`, `key`, `rows`, `extra`) are primary evidence of plan improvements.

In [10]:
import json
import random
import statistics
import sys
import time
from pathlib import Path
from urllib import request as urlrequest
from urllib.error import URLError

from IPython.display import Markdown, display

# Make app modules importable when notebook runs from Module_B/.
APP_DIR = Path.cwd() / "app"
if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))

from database import get_db_connection, DB_HOST, DB_USER, DB_NAME

BENCH_PREFIX = "[BENCH]"
TARGET_BENCH_POSTS = 3000
COMMENTS_PER_BENCH_POST = 3
TARGET_HOT_POST_COMMENTS = 2500
ITERATIONS = 40
WARMUP = 5
DISCOVERY_ITERATIONS = 10
DISCOVERY_WARMUP = 2
API_BASE = "http://127.0.0.1:8001"

LIST_POSTS_SQL = """
    SELECT
        p.PostID,
        p.MemberID,
        m.Name AS AuthorName,
        p.Content,
        p.MediaURL,
        p.MediaType,
        p.PostDate,
        p.LastEditDate,
        p.Visibility,
        p.LikeCount,
        p.CommentCount
    FROM Post p
    JOIN Member m ON p.MemberID = m.MemberID
    WHERE p.IsActive = TRUE
    ORDER BY p.PostDate DESC, p.PostID DESC
    LIMIT %s OFFSET %s
"""

LIST_COMMENTS_SQL = """
    SELECT
        c.CommentID,
        c.PostID,
        c.MemberID,
        m.Name AS AuthorName,
        c.Content,
        c.CommentDate,
        c.LastEditDate,
        c.LikeCount,
        c.IsActive
    FROM Comment c
    JOIN Member m ON c.MemberID = m.MemberID
    WHERE c.PostID = %s AND c.IsActive = TRUE
    ORDER BY c.CommentDate ASC
"""

BASELINE_INDEX_DEFS = [
    ("Post", "idx_post_member", "CREATE INDEX idx_post_member ON Post(MemberID)"),
    ("Comment", "idx_comment_post", "CREATE INDEX idx_comment_post ON Comment(PostID)"),
    ("Comment", "idx_comment_member", "CREATE INDEX idx_comment_member ON Comment(MemberID)"),
]

BEFORE_STAGE_DROP_INDEXES = [
    ("Post", "idx_post_date"),
    ("Post", "idx_post_active_postdate"),
    ("Post", "idx_post_active_postdate_postid"),
    ("Post", "idx_post_active_date_member"),
    ("Post", "idx_post_date_active"),
    ("Comment", "idx_comment_post_active_date"),
]

AFTER_STAGE_INDEX_DEFS = {
    "list_posts": (
        "Post",
        "idx_post_active_postdate_postid",
        "CREATE INDEX idx_post_active_postdate_postid ON Post(IsActive, PostDate DESC, PostID DESC)",
    ),
    "list_comments": (
        "Comment",
        "idx_comment_post_active_date",
        "CREATE INDEX idx_comment_post_active_date ON Comment(PostID, IsActive, CommentDate ASC)",
    ),
}


def safe_drop_index(cursor, table_name, index_name):
    cursor.execute(
        """
        SELECT COUNT(*) AS c
        FROM information_schema.statistics
        WHERE table_schema = DATABASE() AND table_name = %s AND index_name = %s
        """,
        (table_name, index_name),
    )
    if cursor.fetchone()["c"] > 0:
        cursor.execute(f"ALTER TABLE `{table_name}` DROP INDEX `{index_name}`")


def safe_create_index(cursor, table_name, index_name, ddl):
    cursor.execute(
        """
        SELECT COUNT(*) AS c
        FROM information_schema.statistics
        WHERE table_schema = DATABASE() AND table_name = %s AND index_name = %s
        """,
        (table_name, index_name),
    )
    if cursor.fetchone()["c"] == 0:
        cursor.execute(ddl)


def ensure_benchmark_data():
    conn = get_db_connection()
    try:
        with conn.cursor() as cursor:
            cursor.execute(
                "SELECT COUNT(*) AS c FROM Post WHERE Content LIKE %s",
                (f"{BENCH_PREFIX}%",),
            )
            existing_posts = cursor.fetchone()["c"]

            to_add = max(0, TARGET_BENCH_POSTS - existing_posts)
            if to_add > 0:
                post_values = [
                    (1, f"{BENCH_PREFIX} synthetic post {existing_posts + i + 1}", None, "None", "Public")
                    for i in range(to_add)
                ]
                cursor.executemany(
                    """
                    INSERT INTO Post (MemberID, Content, MediaURL, MediaType, Visibility)
                    VALUES (%s, %s, %s, %s, %s)
                    """,
                    post_values,
                )

                cursor.execute(
                    """
                    SELECT PostID
                    FROM Post
                    WHERE Content LIKE %s
                    ORDER BY PostID DESC
                    LIMIT %s
                    """,
                    (f"{BENCH_PREFIX}%", to_add),
                )
                inserted_posts = [row["PostID"] for row in cursor.fetchall()]

                comment_values = []
                for post_id in inserted_posts:
                    for j in range(COMMENTS_PER_BENCH_POST):
                        comment_values.append((post_id, 1, f"{BENCH_PREFIX} synthetic comment {j + 1} for post {post_id}"))
                cursor.executemany(
                    """
                    INSERT INTO Comment (PostID, MemberID, Content)
                    VALUES (%s, %s, %s)
                    """,
                    comment_values,
                )

            cursor.execute("SELECT PostID FROM Post WHERE Content = %s LIMIT 1", (f"{BENCH_PREFIX} hotspot post",))
            hotspot = cursor.fetchone()
            if hotspot:
                hotspot_post_id = hotspot["PostID"]
            else:
                cursor.execute(
                    """
                    INSERT INTO Post (MemberID, Content, MediaURL, MediaType, Visibility)
                    VALUES (%s, %s, %s, %s, %s)
                    """,
                    (1, f"{BENCH_PREFIX} hotspot post", None, "None", "Public"),
                )
                hotspot_post_id = cursor.lastrowid

            cursor.execute(
                """
                SELECT COUNT(*) AS c
                FROM Comment
                WHERE PostID = %s AND Content LIKE %s
                """,
                (hotspot_post_id, f"{BENCH_PREFIX} hotspot comment%"),
            )
            existing_hot_comments = cursor.fetchone()["c"]
            to_add_hot = max(0, TARGET_HOT_POST_COMMENTS - existing_hot_comments)
            if to_add_hot > 0:
                hot_comment_values = [
                    (hotspot_post_id, 1, f"{BENCH_PREFIX} hotspot comment {existing_hot_comments + i + 1}")
                    for i in range(to_add_hot)
                ]
                cursor.executemany(
                    """
                    INSERT INTO Comment (PostID, MemberID, Content)
                    VALUES (%s, %s, %s)
                    """,
                    hot_comment_values,
                )

            return hotspot_post_id
    finally:
        conn.close()


def percentile(values, pct):
    idx = int(round((pct / 100.0) * (len(values) - 1)))
    return sorted(values)[idx]


def summarize_times(values):
    return {
        "avg_ms": round(statistics.mean(values), 3),
        "median_ms": round(statistics.median(values), 3),
        "p95_ms": round(percentile(values, 95), 3),
        "min_ms": round(min(values), 3),
        "max_ms": round(max(values), 3),
    }


def run_sql_timing(sql, params_builder, *, warmup, iterations):
    conn = get_db_connection()
    durations = []
    try:
        with conn.cursor() as cursor:
            for _ in range(warmup):
                cursor.execute(sql, params_builder())
                cursor.fetchall()
            for _ in range(iterations):
                start = time.perf_counter()
                cursor.execute(sql, params_builder())
                cursor.fetchall()
                durations.append((time.perf_counter() - start) * 1000.0)
    finally:
        conn.close()
    return durations


def get_explain(sql, params):
    conn = get_db_connection()
    try:
        with conn.cursor() as cursor:
            cursor.execute("EXPLAIN " + sql, params)
            rows = cursor.fetchall()
    finally:
        conn.close()
    return [
        {
            "table": r.get("table"),
            "type": r.get("type"),
            "key": r.get("key"),
            "rows": r.get("rows"),
            "extra": r.get("Extra"),
        }
        for r in rows
    ]


def run_api_timing(limit, offset, post_id, *, warmup, iterations):
    login_req = urlrequest.Request(
        f"{API_BASE}/login",
        data=json.dumps({"username": "rahul.sharma@iitgn.ac.in", "password": "password123"}).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urlrequest.urlopen(login_req, timeout=20) as resp:
        token = json.loads(resp.read().decode("utf-8"))["session_token"]

    headers = {"session-token": token, "Content-Type": "application/json"}

    def api_get(path):
        req = urlrequest.Request(f"{API_BASE}{path}", headers=headers, method="GET")
        with urlrequest.urlopen(req, timeout=20) as resp:
            payload = resp.read().decode("utf-8")
            return resp.status, payload

    posts_times = []
    comments_times = []

    for _ in range(warmup):
        api_get(f"/posts?limit={limit}&offset={offset}")
        api_get(f"/posts/{post_id}/comments")

    for _ in range(iterations):
        start = time.perf_counter()
        st, pl = api_get(f"/posts?limit={limit}&offset={offset}")
        posts_times.append((time.perf_counter() - start) * 1000.0)
        if st != 200:
            raise RuntimeError(f"/posts failed: {st} {pl}")

        start = time.perf_counter()
        st, pl = api_get(f"/posts/{post_id}/comments")
        comments_times.append((time.perf_counter() - start) * 1000.0)
        if st != 200:
            raise RuntimeError(f"/posts/{{id}}/comments failed: {st} {pl}")

    return posts_times, comments_times


def choose_benchmark_params():
    conn = get_db_connection()
    try:
        with conn.cursor() as cursor:
            cursor.execute("SELECT COUNT(*) AS c FROM Post WHERE IsActive = TRUE")
            total_active_posts = cursor.fetchone()["c"]
    finally:
        conn.close()

    limit = 20
    offset = min(total_active_posts - limit, total_active_posts // 2) if total_active_posts > limit else 0
    return limit, offset


def set_indexes(enabled, selected_benchmarks=None):
    selected = set(selected_benchmarks or AFTER_STAGE_INDEX_DEFS.keys())
    conn = get_db_connection()
    try:
        with conn.cursor() as cursor:
            for table_name, index_name, ddl in BASELINE_INDEX_DEFS:
                safe_create_index(cursor, table_name, index_name, ddl)
            for table_name, index_name in BEFORE_STAGE_DROP_INDEXES:
                safe_drop_index(cursor, table_name, index_name)
            if enabled:
                for benchmark_key, (table_name, index_name, ddl) in AFTER_STAGE_INDEX_DEFS.items():
                    if benchmark_key in selected:
                        safe_create_index(cursor, table_name, index_name, ddl)
    finally:
        conn.close()


def run_stage(stage_name, index_enabled, limit, offset, post_id, selected_benchmarks):
    set_indexes(index_enabled, selected_benchmarks)
    posts_sql_times = run_sql_timing(LIST_POSTS_SQL, lambda: (limit, offset), warmup=WARMUP, iterations=ITERATIONS)
    comments_sql_times = run_sql_timing(LIST_COMMENTS_SQL, lambda: (post_id,), warmup=WARMUP, iterations=ITERATIONS)
    posts_api_times, comments_api_times = run_api_timing(limit, offset, post_id, warmup=WARMUP, iterations=ITERATIONS)

    return {
        "stage": stage_name,
        "indexes_enabled": index_enabled,
        "sql_ms": {
            "list_posts": summarize_times(posts_sql_times),
            "list_comments": summarize_times(comments_sql_times),
        },
        "api_ms": {
            "list_posts": summarize_times(posts_api_times),
            "list_comments": summarize_times(comments_api_times),
        },
        "explain": {
            "list_posts": get_explain(LIST_POSTS_SQL, (limit, offset)),
            "list_comments": get_explain(LIST_COMMENTS_SQL, (post_id,)),
        },
        "selected_benchmarks": selected_benchmarks,
    }


def fmt(value):
    return "-" if value is None else f"{value:.3f}"


def speedup(before_ms, after_ms):
    if after_ms == 0:
        return None
    return round(before_ms / after_ms, 3)


try:
    urlrequest.urlopen(f"{API_BASE}/docs", timeout=5)
    display(Markdown("### Environment check\n- API server reachable at `http://127.0.0.1:8001`\n- Benchmark setup loaded"))
except URLError as exc:
    raise RuntimeError(
        "API server is not reachable. Start it with: uvicorn main:app --reload --port 8001 (from Module_B/app)."
    ) from exc

### Environment check
- API server reachable at `http://127.0.0.1:8001`
- Benchmark setup loaded

In [11]:
# Step 1 and 2: Run tests without indexing and identify slow endpoints.
hotspot_post_id = ensure_benchmark_data()
limit, offset = choose_benchmark_params()
post_id = hotspot_post_id

# Force strict baseline (no optimization indexes).
set_indexes(False)

posts_discovery = run_sql_timing(
    LIST_POSTS_SQL,
    lambda: (limit, offset),
    warmup=DISCOVERY_WARMUP,
    iterations=DISCOVERY_ITERATIONS,
)
comments_discovery = run_sql_timing(
    LIST_COMMENTS_SQL,
    lambda: (post_id,),
    warmup=DISCOVERY_WARMUP,
    iterations=DISCOVERY_ITERATIONS,
)

discovery = [
    {"benchmark": "list_posts", "sql_avg_ms": round(statistics.mean(posts_discovery), 3)},
    {"benchmark": "list_comments", "sql_avg_ms": round(statistics.mean(comments_discovery), 3)},
]
discovery.sort(key=lambda x: x["sql_avg_ms"], reverse=True)
auto_slowest = [d["benchmark"] for d in discovery]

before = run_stage("before_indexes", False, limit, offset, post_id, auto_slowest)

headers = ["Benchmark", "Pre-index SQL Avg (ms)"]
rows = [
    "| " + " | ".join(headers) + " |",
    "|" + "|".join(["---"] * len(headers)) + "|",
]
for d in discovery:
    rows.append(f"| {d['benchmark']} | {fmt(d['sql_avg_ms'])} |")

display(Markdown("### Step 1: Baseline run complete (without indexing)"))
display(Markdown(f"- limit: `{limit}`\n- offset: `{offset}`\n- hotspot post id: `{post_id}`"))
display(Markdown("### Step 2: Auto-identified slow endpoints\n\n" + "\n".join(rows)))

before

### Step 1: Baseline run complete (without indexing)

- limit: `20`
- offset: `1510`
- hotspot post id: `3021`

### Step 2: Auto-identified slow endpoints

| Benchmark | Pre-index SQL Avg (ms) |
|---|---|
| list_comments | 24.073 |
| list_posts | 5.580 |

{'stage': 'before_indexes',
 'indexes_enabled': False,
 'sql_ms': {'list_posts': {'avg_ms': 5.6,
   'median_ms': 4.604,
   'p95_ms': 10.427,
   'min_ms': 3.176,
   'max_ms': 10.709},
  'list_comments': {'avg_ms': 23.087,
   'median_ms': 22.492,
   'p95_ms': 28.352,
   'min_ms': 20.484,
   'max_ms': 29.759}},
 'api_ms': {'list_posts': {'avg_ms': 37.851,
   'median_ms': 38.02,
   'p95_ms': 48.853,
   'min_ms': 16.582,
   'max_ms': 58.822},
  'list_comments': {'avg_ms': 95.215,
   'median_ms': 94.93,
   'p95_ms': 111.688,
   'min_ms': 74.243,
   'max_ms': 123.965}},
 'explain': {'list_posts': [{'table': 'p',
    'type': 'ALL',
    'key': None,
    'rows': 3021,
    'extra': 'Using where; Using filesort'},
   {'table': 'm',
    'type': 'eq_ref',
    'key': 'PRIMARY',
    'rows': 1,
    'extra': None}],
  'list_comments': [{'table': 'c',
    'type': 'ref',
    'key': 'idx_comment_post',
    'rows': 2500,
    'extra': 'Using where; Using filesort'},
   {'table': 'm',
    'type': 'eq_ref',
  

In [12]:
# Step 3 and 4: Apply indexes for slow endpoints, rerun, and compare.
after = run_stage("after_indexes", True, limit, offset, post_id, auto_slowest)

speedup_summary = {
    "posts_sql_speedup": speedup(before["sql_ms"]["list_posts"]["avg_ms"], after["sql_ms"]["list_posts"]["avg_ms"]),
    "comments_sql_speedup": speedup(before["sql_ms"]["list_comments"]["avg_ms"], after["sql_ms"]["list_comments"]["avg_ms"]),
    "posts_api_speedup": speedup(before["api_ms"]["list_posts"]["avg_ms"], after["api_ms"]["list_posts"]["avg_ms"]),
    "comments_api_speedup": speedup(before["api_ms"]["list_comments"]["avg_ms"], after["api_ms"]["list_comments"]["avg_ms"]),
}

comparison_rows = [
    {
        "Endpoint": "GET /posts",
        "Layer": "SQL",
        "Before Avg (ms)": before["sql_ms"]["list_posts"]["avg_ms"],
        "After Avg (ms)": after["sql_ms"]["list_posts"]["avg_ms"],
        "Speedup (x)": speedup_summary["posts_sql_speedup"],
    },
    {
        "Endpoint": "GET /posts/{post_id}/comments",
        "Layer": "SQL",
        "Before Avg (ms)": before["sql_ms"]["list_comments"]["avg_ms"],
        "After Avg (ms)": after["sql_ms"]["list_comments"]["avg_ms"],
        "Speedup (x)": speedup_summary["comments_sql_speedup"],
    },
    {
        "Endpoint": "GET /posts",
        "Layer": "API",
        "Before Avg (ms)": before["api_ms"]["list_posts"]["avg_ms"],
        "After Avg (ms)": after["api_ms"]["list_posts"]["avg_ms"],
        "Speedup (x)": speedup_summary["posts_api_speedup"],
    },
    {
        "Endpoint": "GET /posts/{post_id}/comments",
        "Layer": "API",
        "Before Avg (ms)": before["api_ms"]["list_comments"]["avg_ms"],
        "After Avg (ms)": after["api_ms"]["list_comments"]["avg_ms"],
        "Speedup (x)": speedup_summary["comments_api_speedup"],
    },
]

headers = ["Endpoint", "Layer", "Before Avg (ms)", "After Avg (ms)", "Speedup (x)"]
comp_table = [
    "| " + " | ".join(headers) + " |",
    "|" + "|".join(["---"] * len(headers)) + "|",
]
for row in comparison_rows:
    comp_table.append(
        "| "
        + " | ".join(
            [
                str(row["Endpoint"]),
                str(row["Layer"]),
                fmt(row["Before Avg (ms)"]),
                fmt(row["After Avg (ms)"]),
                fmt(row["Speedup (x)"]),
            ]
        )
        + " |"
    )


def explain_rows(query_name, stage_name, data_rows):
    rows = []
    for r in data_rows:
        rows.append(
            {
                "Query": query_name,
                "Stage": stage_name,
                "Table": r.get("table"),
                "Type": r.get("type"),
                "Key": r.get("key"),
                "Rows": r.get("rows"),
                "Extra": r.get("extra"),
            }
        )
    return rows

explain_all = []
explain_all.extend(explain_rows("GET /posts", "before", before["explain"]["list_posts"]))
explain_all.extend(explain_rows("GET /posts", "after", after["explain"]["list_posts"]))
explain_all.extend(explain_rows("GET /posts/{post_id}/comments", "before", before["explain"]["list_comments"]))
explain_all.extend(explain_rows("GET /posts/{post_id}/comments", "after", after["explain"]["list_comments"]))

ex_headers = ["Query", "Stage", "Table", "Type", "Key", "Rows", "Extra"]
ex_table = [
    "| " + " | ".join(ex_headers) + " |",
    "|" + "|".join(["---"] * len(ex_headers)) + "|",
]
for row in explain_all:
    ex_table.append(
        "| "
        + " | ".join(
            [
                str(row["Query"]),
                str(row["Stage"]),
                str(row["Table"]),
                str(row["Type"]),
                str(row["Key"]),
                str(row["Rows"]),
                str(row["Extra"]),
            ]
        )
        + " |"
    )

results = {
    "db_config": {
        "host": DB_HOST,
        "user": DB_USER,
        "database": DB_NAME,
    },
    "benchmark_params": {
        "iterations": ITERATIONS,
        "warmup": WARMUP,
        "limit": limit,
        "offset": offset,
        "comment_post_id": post_id,
        "auto_identified_slowest": auto_slowest,
    },
    "discovery": discovery,
    "stages": [before, after],
    "speedup": speedup_summary,
}

out_dir = Path("performance")
out_dir.mkdir(parents=True, exist_ok=True)
out_file = out_dir / "index_benchmark_results.json"
out_file.write_text(json.dumps(results, indent=2), encoding="utf-8")

display(Markdown("### Step 3: Indexes applied to identified slow endpoints\n\n- " + "\n- ".join(auto_slowest)))
display(Markdown("### Step 4: Before vs After comparison\n\n" + "\n".join(comp_table)))
display(Markdown("### EXPLAIN plan comparison\n\n" + "\n".join(ex_table)))
display(Markdown(f"Saved artifact: `performance/index_benchmark_results.json`"))

results

### Step 3: Indexes applied to identified slow endpoints

- list_comments
- list_posts

### Step 4: Before vs After comparison

| Endpoint | Layer | Before Avg (ms) | After Avg (ms) | Speedup (x) |
|---|---|---|---|---|
| GET /posts | SQL | 5.600 | 5.528 | 1.013 |
| GET /posts/{post_id}/comments | SQL | 23.087 | 20.358 | 1.134 |
| GET /posts | API | 37.851 | 35.997 | 1.052 |
| GET /posts/{post_id}/comments | API | 95.215 | 91.577 | 1.040 |

### EXPLAIN plan comparison

| Query | Stage | Table | Type | Key | Rows | Extra |
|---|---|---|---|---|---|---|
| GET /posts | before | p | ALL | None | 3021 | Using where; Using filesort |
| GET /posts | before | m | eq_ref | PRIMARY | 1 | None |
| GET /posts | after | p | ALL | None | 3021 | Using where; Using filesort |
| GET /posts | after | m | eq_ref | PRIMARY | 1 | None |
| GET /posts/{post_id}/comments | before | c | ref | idx_comment_post | 2500 | Using where; Using filesort |
| GET /posts/{post_id}/comments | before | m | eq_ref | PRIMARY | 1 | None |
| GET /posts/{post_id}/comments | after | c | range | idx_comment_post_active_date | 2500 | Using index condition |
| GET /posts/{post_id}/comments | after | m | eq_ref | PRIMARY | 1 | None |

Saved artifact: `performance/index_benchmark_results.json`

{'db_config': {'host': 'localhost',
  'user': 'root',
  'database': 'college_social_media'},
 'benchmark_params': {'iterations': 40,
  'warmup': 5,
  'limit': 20,
  'offset': 1510,
  'comment_post_id': 3021,
  'auto_identified_slowest': ['list_comments', 'list_posts']},
 'discovery': [{'benchmark': 'list_comments', 'sql_avg_ms': 24.073},
  {'benchmark': 'list_posts', 'sql_avg_ms': 5.58}],
 'stages': [{'stage': 'before_indexes',
   'indexes_enabled': False,
   'sql_ms': {'list_posts': {'avg_ms': 5.6,
     'median_ms': 4.604,
     'p95_ms': 10.427,
     'min_ms': 3.176,
     'max_ms': 10.709},
    'list_comments': {'avg_ms': 23.087,
     'median_ms': 22.492,
     'p95_ms': 28.352,
     'min_ms': 20.484,
     'max_ms': 29.759}},
   'api_ms': {'list_posts': {'avg_ms': 37.851,
     'median_ms': 38.02,
     'p95_ms': 48.853,
     'min_ms': 16.582,
     'max_ms': 58.822},
    'list_comments': {'avg_ms': 95.215,
     'median_ms': 94.93,
     'p95_ms': 111.688,
     'min_ms': 74.243,
     'max_

## 5) Video Demonstration Link

Add your hosted 3-5 minute demo link here (YouTube unlisted or open Drive link):
- `<PASTE_VIDEO_LINK_HERE>`

## 6) Conclusion

- Module B integrates secure local APIs, RBAC enforcement, and auditable DB operations.
- Indexing/benchmarking pipeline is implemented with before-vs-after quantitative evidence.
- Remaining result analysis can be expanded with larger datasets if needed.